## Sequential Workflow with MAF and Microsoft Foundry

**Fundamental logic:** A sequential workflow passes one stage?s output into the next stage in a fixed order.


![sequential_workflow](./Assets/sequential_workflow.png)

**Fundamental logic:** The diagram shows a simple pipeline where research must finish before writing begins.


In [1]:
# Fundamental logic: Pinned packages ensure the notebook uses compatible Agent Framework and Azure SDK versions.

%pip install agent-framework==1.0.0b251209 python-dotenv azure-ai-projects==2.0.0b2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setting Up the Environment

**Fundamental logic:** The setup step loads the Foundry project endpoint and selected model deployment.


In [2]:
# Fundamental logic: Environment-based configuration makes the notebook portable between projects without code
# changes.

import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

Project Endpoint:  https://ajay-agent-project111-resource.services.ai.azure.com/api/projects/ajay-agent-project111
Model:  ajay-gpt-4o


### Defining the Function to Create a Chat Agent

**Fundamental logic:** The helper centralizes the repeated steps required to create each conversation-scoped specialist agent.


In [3]:
# Fundamental logic: Async network calls authenticate, create a conversation, bind a model, and return an agent
# configured by its instructions.

from agent_framework import ChatAgent
from agent_framework.azure import AzureAIClient
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import AzureCliCredential

# Cell 2: Define async workflow
async def create_agent(agent_name: str,
                       agent_instructions: str) -> ChatAgent:
    
    # Create async Azure credential
    credential = AzureCliCredential()

    # creating the Foundry Project Client
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=credential
    )

    # creating a conversation using the OpenAI Client
    openai_client = project_client.get_openai_client()
    conversation = await openai_client.conversations.create()
    conversation_id = conversation.id
    print("Conversation ID: ", conversation_id)
    
    # Initialize the Azure AI Agent Client
    chat_client = AzureAIClient(project_client=project_client,
                                conversation_id=conversation_id,
                                model_deployment_name=model)

    try:
        agent = chat_client.create_agent(
            name=agent_name,
            instructions=agent_instructions,
        )

        print("{} Agent created successfully!".format(agent_name))
        return agent

    finally:
        # Clean up async clients
        await chat_client.close()
        await credential.close()

### Creating the Researcher Agent

**Fundamental logic:** The researcher is responsible only for producing factual source material for the next stage.


In [4]:
# Fundamental logic: This role separation makes the researcher?s response a clear intermediate workflow artifact.

researcher_agent = await create_agent(
    agent_name="Researcher-Agent",
    agent_instructions="You are a knowledgeable researcher. Your task is to gather information and provide insights on a given topic. "
                       "You should use reliable sources and present the information in a clear and concise manner."
    )

Conversation ID:  conv_970ed85ad9b502f600PIYS0YlZ8i3k9L7s8FXUaf7qVp5PFJwM
Researcher-Agent Agent created successfully!


### Creating the Writer Agent

**Fundamental logic:** The writer transforms the researcher?s output into a coherent final essay.


In [5]:
# Fundamental logic: The writer does not receive the original prompt directly; it receives the upstream research
# message.

writer_agent = await create_agent(
    agent_name="Writer-Agent",
    agent_instructions="You are a creative writer. Your task is to write an essay on a given topic. "
                       "You should focus on clarity, coherence, and engaging storytelling."
    )

Conversation ID:  conv_587877eb97e4ff4400bkYVfR783GxqTiOpLlKomgpbsi5JAaaz
Writer-Agent Agent created successfully!


### Creating the First Executor to run the Researcher Agent

**Fundamental logic:** An executor function is a workflow node that receives typed input and communicates through WorkflowContext.


In [6]:
# Fundamental logic: send_message publishes an intermediate result and activates the downstream writer node.

from agent_framework import WorkflowBuilder, WorkflowContext, WorkflowOutputEvent, executor
from typing import Any
from agent_framework import ChatResponse

@executor(id = "run_researcher_agent")
async def run_researcher_agent(query: str,
                               ctx: WorkflowContext[str]) -> None:
    response = await researcher_agent.run(query)

    await ctx.send_message(str(response))

### Creating the Second Executor to run the Writer Agent

**Fundamental logic:** The second executor is the terminal stage of the pipeline.


In [7]:
# Fundamental logic: yield_output marks the writer response as the workflow?s final externally visible result.

@executor(id = "run_writer_agent")
async def run_writer_agent(research_data: str,
                          ctx: WorkflowContext[str]) -> None:
    response = await writer_agent.run(research_data)

    await ctx.yield_output(str(response))

### Creating the Sequential Workflow

**Fundamental logic:** A directed edge establishes both execution order and the path used to pass data between stages.


In [8]:
# Fundamental logic: The researcher is the start node, and its single outgoing edge guarantees that the writer runs
# second.

from agent_framework import WorkflowBuilder, WorkflowViz

workflow = (
    WorkflowBuilder()
    .add_edge(run_researcher_agent, run_writer_agent)
    .set_start_executor(run_researcher_agent)
    .build()
)

viz = WorkflowViz(workflow)

Adding an edge with Executor or AgentProtocol instances directly is not recommended, because workflow instances created from the builder will share the same executor/agent instances. Consider using a registered name for lazy initialization instead.


### Generating Mermaid Diagram for Visualization

**Fundamental logic:** Mermaid visualization provides a quick structural check of the workflow graph.


In [9]:
# Fundamental logic: Rendering the generated graph confirms that the workflow is a two-node sequential pipeline.

mermaid_content = viz.to_mermaid()

# printing mermaid content as markdown
from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{mermaid_content}\n```"))

```mermaid
flowchart TD
  run_researcher_agent["run_researcher_agent (Start)"];
  run_writer_agent["run_writer_agent"];
  internal_run_researcher_agent --> run_researcher_agent;
  internal_run_writer_agent --> run_writer_agent;
  run_researcher_agent --> run_writer_agent;
```

### Running the Workflow and Streaming Events

**Fundamental logic:** Workflow streaming reveals lifecycle events and the final output while the graph runs.


In [10]:
# Fundamental logic: run_stream starts with the user prompt; the final WorkflowOutputEvent contains the completed
# essay.

# Run the workflow and stream events in notebook
async def main():
    async for event in workflow.run_stream("write an essay about the impact of AI on society"):
        print(f"Event: {event}")
        if isinstance(event, WorkflowOutputEvent):
            print(f"Workflow completed with result: {event.data}")

await main()

Event: WorkflowStartedEvent(origin=WorkflowEventSource.FRAMEWORK, data=None)
Event: WorkflowStatusEvent(state=WorkflowRunState.IN_PROGRESS, data=None, origin=WorkflowEventSource.FRAMEWORK)
Event: ExecutorInvokedEvent(executor_id=run_researcher_agent, data=write an essay about the impact of AI on society)
Event: ExecutorCompletedEvent(executor_id=run_researcher_agent, data=['**The Impact of Artificial Intelligence on Society**\n\nArtificial Intelligence (AI) has emerged as one of the most transformative technologies of the 21st century, driving unprecedented changes across industries and reshaping societal norms. From healthcare to education, transportation to entertainment, AI is revolutionizing how people live, work, and interact. However, while its benefits are vast and promising, AI also raises critical ethical, social, and economic concerns. Understanding its multifaceted impact is essential for navigating a future in which AI plays a central role in society.\n\n### **Positive Impa